In [ ]:
import csv
import glob
import os

import numpy as np
from netCDF4 import Dataset
from pathlib import Path

In [24]:
def _load(ds, name):
    if name not in ds.variables:
        raise KeyError(f"variable '{name}' not found (available: {list(ds.variables)})")

    arr = ds.variables[name][:]
    if np.ma.isMaskedArray(arr): # NetCDF marks midding pixels as masked, fill missing values
        arr = arr.filled(-1)  # -1 fails NumCycles>=1 and QA==1
    return np.asarray(arr)

def analyze_pixel_quality(path):
    with Dataset(path, "r") as ds:
        num_cycles_layer = _load(ds, "NumCycles")
        qa_layer = _load(ds, 'QA')

        return np.count_nonzero((num_cycles_layer >= 1) & (qa_layer == 1)), int(num_cycles_layer.size)

In [31]:
sites = ['ARM_Southern_Great_Plains_site', 'Mountainair_Pinyon-Juniper_Woodland', 'NEON_Konza_Prairie_Biological_Station', 'Santa_Rita_Grassland', 'Santa_Rita_Mesquite', 'Sevilleta_shrubland', 'Walnut_Gulch_Kendall_Grasslands', 'Walnut_Gulch_Lucky_Hills_Shrub', 'Willard_Juniper_Savannah', ]
product_dirs = ['PLSP_production_nc', 'PLSP_stage_nc']

dirs = []
for site in sites:
    for product_dir in product_dirs:
        dirs.append(Path('/projectnb/modislc/users/fache/data/planet/') / product_dir / site)

nc_files = []
for d in dirs:
    if os.path.isdir(d):
        nc_files.extend(glob.glob(os.path.join(d, "*.nc"), recursive=True))
nc_files = sorted(set(nc_files))

In [34]:
results = []
for path in nc_files:
    print(f"analyzing: {path}")
    good_pixels, total_pixels = analyze_pixel_quality(path)
    percent_good_pixels = 100.0 * good_pixels / total_pixels
    results.append({
        "file": path,
        "site": Path(path).parent.name,
        "year": Path(path).stem.split("_")[-1],
        "good_pixels": good_pixels,
        "total_pixels": total_pixels,
        "percent_good_pixels": f"{percent_good_pixels:.3f}%"
    })

analyzing: /projectnb/modislc/users/fache/data/planet/PLSP_production_nc/ARM_Southern_Great_Plains_site/US-ARM_ARM_Southern_Great_Plains_site_PLSP_2017.nc
analyzing: /projectnb/modislc/users/fache/data/planet/PLSP_production_nc/ARM_Southern_Great_Plains_site/US-ARM_ARM_Southern_Great_Plains_site_PLSP_2018.nc
analyzing: /projectnb/modislc/users/fache/data/planet/PLSP_production_nc/ARM_Southern_Great_Plains_site/US-ARM_ARM_Southern_Great_Plains_site_PLSP_2019.nc
analyzing: /projectnb/modislc/users/fache/data/planet/PLSP_production_nc/ARM_Southern_Great_Plains_site/US-ARM_ARM_Southern_Great_Plains_site_PLSP_2020.nc
analyzing: /projectnb/modislc/users/fache/data/planet/PLSP_production_nc/ARM_Southern_Great_Plains_site/US-ARM_ARM_Southern_Great_Plains_site_PLSP_2021.nc
analyzing: /projectnb/modislc/users/fache/data/planet/PLSP_production_nc/Mountainair_Pinyon-Juniper_Woodland/US-Mpj_Mountainair_Pinyon-Juniper_Woodland_PLSP_2017.nc
analyzing: /projectnb/modislc/users/fache/data/planet/PLSP_p

In [35]:
with open("/projectnb/modislc/users/fache/src/PLSP/code/code_lsp_analysis/check_site_pixel_quality_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["file", "site", "year", "good_pixels", "total_pixels", "percent_good_pixels"])
    writer.writeheader()
    writer.writerows(results)